# COMI training on RunPod
Use a Python 3 notebook kernel. Upload this notebook and `comi-runpod.zip` to `/workspace`, then execute cells in order.

Training follows the repo: full encoder (8 layers), full memory fusion (1 layer), decoder LoRA (rank 128). The training environment is isolated from the notebook kernel; no kernel switch or `!source` is needed.

Default segment size is 4096. Setting it to 1024 splits context into segments; it does not truncate examples to 1024 tokens. Full training starts as a background process with logs on disk, so it can continue when the browser closes. Stopping the Pod stops training.


## 1. Paths and training settings

In [ ]:
from pathlib import Path
import os
import json
import subprocess
import shutil
import zipfile
from collections import deque

REPO = Path('/workspace/comi')
ARCHIVE = Path('/workspace/comi-runpod.zip')
DATA = Path('/workspace/comi-data/hotpotqa')
OUTPUT = Path('/workspace/comi-output/hotpotqa')
SMOKE_OUTPUT = Path('/workspace/comi-output/hotpotqa-smoke')
LOG_DIR = Path('/workspace/comi-logs')
VENV = Path('/workspace/comi-venv')

MODEL_NAME = 'Qwen/Qwen3-4B-Instruct-2507'
NUM_GPUS = 1
SEGMENT_SIZE = 4096
MERGE_SIZE = 16
MAX_STEPS = 10000  # About 0.884 epoch at effective batch 8; 11306 steps is about one epoch.
GRADIENT_ACCUMULATION_STEPS = 8

if not (REPO / 'scripts/runpod_train.sh').exists():
    assert ARCHIVE.is_file(), f'Upload ZIP to {ARCHIVE}'
    REPO.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as archive:
        for entry in archive.infolist():
            target = (REPO / entry.filename).resolve()
            assert target.is_relative_to(REPO.resolve()), 'Invalid archive path'
        archive.extractall(REPO)
print('Repository:', REPO)
subprocess.run(['nvidia-smi'], check=True)


## 2. Install the training environment (first run only)
Uses PyTorch 2.8.0/CUDA 12.9 for Blackwell and disables inherited hf_transfer settings. Installation runs a real BF16 CUDA kernel check.


In [ ]:
subprocess.run(['bash', 'scripts/runpod_setup.sh'], cwd=REPO, check=True)
PYTHON = VENV / 'bin/python'
ENV = os.environ.copy()
ENV.update({
    'PATH': str(VENV / 'bin') + os.pathsep + ENV['PATH'],
    'HF_HOME': '/workspace/hf-cache',
    'HF_HUB_ENABLE_HF_TRANSFER': '0',
    'TOKENIZERS_PARALLELISM': 'false',
    'PYTHONUNBUFFERED': '1',
    'DATA_DIR': str(DATA),
    'TRAIN_FILE': str(DATA / 'train.jsonl'),
    'TEST_FILE': str(DATA / 'eval.jsonl'),
    'MODEL_NAME': MODEL_NAME,
    'NUM_GPUS': str(NUM_GPUS),
    'SEGMENT_SIZE': str(SEGMENT_SIZE),
    'MERGE_SIZE': str(MERGE_SIZE),
    'MAX_STEPS': str(MAX_STEPS),
    'GRADIENT_ACCUMULATION_STEPS': str(GRADIENT_ACCUMULATION_STEPS),
    'OUTPUT_DIR': str(OUTPUT),
    'SMOKE_OUTPUT_DIR': str(SMOKE_OUTPUT),
})
subprocess.run([str(PYTHON), '-m', 'pytest', '-q'], cwd=REPO, env=ENV, check=True)


## 3. Copy prepared data and verify its hashes
No dataset download is needed when using the provided ZIP. Existing data is verified instead of overwritten.

In [ ]:
import hashlib
if not DATA.exists():
    shutil.copytree(REPO / 'data/hotpotqa', DATA)
manifest = json.loads((DATA / 'manifest.json').read_text())
for name, info in manifest['files'].items():
    digest = hashlib.sha256()
    with (DATA / name).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    assert digest.hexdigest() == info['sha256'], f'Checksum mismatch: {name}'
    print(name, info['rows'], 'rows: OK')
subprocess.run([str(PYTHON), 'scripts/preflight.py',
                '--train-file', str(DATA / 'train.jsonl'),
                '--eval-file', str(DATA / 'eval.jsonl'),
                '--num-gpus', str(NUM_GPUS), '--output-dir', str(OUTPUT)],
               cwd=REPO, env=ENV, check=True)


## 4. Real-model smoke training
Downloads Qwen weights on first run. Trains 3 steps and saves a checkpoint. Inspect finite loss and GPU memory before proceeding.

In [ ]:
subprocess.run(['bash', 'scripts/runpod_train.sh', 'smoke'], cwd=REPO, env=ENV, check=True)
assert (SMOKE_OUTPUT / 'checkpoint-3/model.safetensors').exists()


## 5. Smoke inference

In [ ]:
smoke_env = ENV.copy()
smoke_env.update({'RESTORE_FROM': str(SMOKE_OUTPUT), 'OUTPUT_DIR': str(SMOKE_OUTPUT),
                  'TEST_FILE': str(DATA / 'smoke_eval.jsonl'), 'NUM_SAMPLES': '10'})
subprocess.run(['bash', 'scripts/runpod_eval.sh'], cwd=REPO, env=smoke_env, check=True)


## 6. Start full training in the background
Run this only after smoke training/inference succeed. The same command automatically resumes an intermediate checkpoint after a Pod restart. Keep the same settings when resuming. This cell starts a paid GPU workload; it returns immediately.

In [ ]:
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG = LOG_DIR / 'hotpotqa-train.log'
PID_FILE = LOG_DIR / 'hotpotqa-train.pid'

def training_is_running():
    if not PID_FILE.exists():
        return False
    pid = int(PID_FILE.read_text().strip())
    proc = Path(f'/proc/{pid}')
    try:
        command = (proc / 'cmdline').read_bytes()
        return b'runpod_train.sh' in command and (proc / 'cwd').resolve() == REPO.resolve()
    except (FileNotFoundError, PermissionError):
        return False

if training_is_running():
    print('Training already running; PID:', PID_FILE.read_text().strip())
else:
    assert (SMOKE_OUTPUT / 'checkpoint-3/model.safetensors').exists(), 'Run smoke test first'
    with LOG.open('ab', buffering=0) as log_handle:
        process = subprocess.Popen(['bash', 'scripts/runpod_train.sh'], cwd=REPO,
                                   env=ENV, stdin=subprocess.DEVNULL,
                                   stdout=log_handle, stderr=subprocess.STDOUT,
                                   start_new_session=True)
    PID_FILE.write_text(str(process.pid))
    print('Training started; PID:', process.pid)
print('Log:', LOG)


## 7. Monitor
Re-run this cell to view progress. A missing process means inspect the log for successful completion or an error; it does not by itself mean success.

In [ ]:
print('Training process active:', training_is_running())
subprocess.run(['nvidia-smi'], check=True)
if LOG.exists():
    with LOG.open(encoding='utf-8', errors='replace') as handle:
        print(''.join(deque(handle, maxlen=60)))
print('Checkpoints:', sorted(path.name for path in OUTPUT.glob('checkpoint-*')))


## 8. Evaluate after training finishes
Runs on all 7,405 validation examples. Results are answer EM/F1, excluding supporting-fact and joint HotpotQA scores.

In [ ]:
assert not training_is_running(), 'Wait for training to finish before evaluation'
assert (OUTPUT / 'model.safetensors').exists(), 'Final model is not available'
state = json.loads((OUTPUT / 'trainer_state.json').read_text())
assert state['global_step'] >= MAX_STEPS, 'Training did not reach the configured number of steps'
eval_env = ENV.copy()
eval_env.update({'RESTORE_FROM': str(OUTPUT), 'OUTPUT_DIR': str(OUTPUT),
                 'TEST_FILE': str(DATA / 'validation.jsonl'), 'NUM_SAMPLES': '0'})
subprocess.run(['bash', 'scripts/runpod_eval.sh'], cwd=REPO, env=eval_env, check=True)
for path in (OUTPUT / 'eval_result').glob('*metrics*.json'):
    print(path.name, json.loads(path.read_text()))


## 9. Save your results
Download `/workspace/comi-output/hotpotqa` and `/workspace/comi-logs` before deleting the Pod. Intermediate `checkpoint-N` directories include optimizer/RNG state for resume; the root contains the final model and COMI configuration.

If the GPU runs out of memory, do not switch training modes accidentally: this notebook retains the original full encoder/fusion and LoRA decoder configuration. Use a GPU with more VRAM, or separately configure optimizer offload and repeat the smoke test.
